# 🎬 Sentiment Analysis Engine — Experimentation Notebook

**A complete, end-to-end NLP & Machine Learning project demonstrating:**
- Text preprocessing pipelines
- TF-IDF feature engineering
- Multiple ML classifiers
- Comprehensive model evaluation
- Interpretability analysis

---

## 📋 Table of Contents
1. [Environment Setup](#1-environment-setup)
2. [Data Loading](#2-data-loading)
3. [Exploratory Data Analysis](#3-eda)
4. [NLP Preprocessing](#4-preprocessing)
5. [Feature Engineering](#5-feature-engineering)
6. [Model Training](#6-model-training)
7. [Model Evaluation](#7-evaluation)
8. [Interpretability](#8-interpretability)
9. [Inference Demo](#9-inference)

## 1. Environment Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))  # add project root

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')

from src.utils import set_seed
set_seed(42)

print('✅ Environment ready')

## 2. Data Loading

We load the IMDB dataset. If no CSV is found in `data/raw/`, a built-in demo dataset is generated automatically so this notebook runs end-to-end without any manual download.

In [ ]:
from src.data_loader import load_dataset, validate_and_clean, compute_statistics, print_eda_summary

# Load (auto-discovers CSV in data/raw/ or generates demo dataset)
df = load_dataset()
df = validate_and_clean(df)
df = compute_statistics(df)

print_eda_summary(df)

## 3. Exploratory Data Analysis (EDA)

> **Why EDA?** Before training any ML model, you must understand your data.
> Skipping EDA leads to biased models, silent bugs, and poor real-world performance.

In [ ]:
# 3.1 — Sentiment Distribution
# ─────────────────────────────────────────────
# Key question: Are classes balanced?
# Imbalanced classes require special handling (oversampling, class_weight, etc.)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
counts = df['sentiment'].value_counts()
colors = ['#2ecc71', '#e74c3c']

axes[0].bar(counts.index, counts.values, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Sentiment Class Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Sentiment')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 50, f'{v:,}', ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=counts.index, colors=colors,
            autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Class Proportion', fontsize=13, fontweight='bold')

plt.suptitle('Class Balance Analysis', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

imbalance_ratio = counts.max() / counts.min()
print(f'\nImbalance ratio: {imbalance_ratio:.2f}x')
print('✅ Dataset is balanced!' if imbalance_ratio < 1.5 else '⚠️ Dataset is imbalanced — consider resampling')

In [ ]:
# 3.2 — Review Length Analysis
# ─────────────────────────────────────────────
# Understanding length distribution helps choose:
#  - max_features in TF-IDF
#  - truncation strategies for deep learning

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for sentiment, color in zip(['positive', 'negative'], ['#2ecc71', '#e74c3c']):
    subset = df[df['sentiment'] == sentiment]['word_count']
    axes[0].hist(subset, bins=50, alpha=0.6, color=color, label=sentiment, edgecolor='white')
    subset.plot.kde(ax=axes[1], color=color, linewidth=2.5, label=sentiment)

axes[0].set_title('Word Count Histogram', fontweight='bold')
axes[0].set_xlabel('Words per Review')
axes[0].legend()

axes[1].set_title('Word Count KDE', fontweight='bold')
axes[1].set_xlabel('Words per Review')
axes[1].legend()

data = [df[df['sentiment']=='positive']['word_count'].values,
        df[df['sentiment']=='negative']['word_count'].values]
bp = axes[2].boxplot(data, patch_artist=True, labels=['positive','negative'])
for patch, color in zip(bp['boxes'], ['#2ecc71', '#e74c3c']):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[2].set_title('Word Count Box Plot', fontweight='bold')

plt.suptitle('Review Length Distribution by Sentiment', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(df.groupby('sentiment')['word_count'].describe().round(1))

In [ ]:
# 3.3 — Word Clouds
# ─────────────────────────────────────────────
from wordcloud import WordCloud

stopwords_set = {'the','a','an','and','or','but','in','on','at','to','for',
                 'of','with','is','was','are','were','this','that','it',
                 'movie','film','br'}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Word Clouds by Sentiment', fontsize=15, fontweight='bold')

for ax, sentiment in zip(axes, ['positive', 'negative']):
    text = ' '.join(df[df['sentiment'] == sentiment]['review'])
    wc = WordCloud(
        width=800, height=400, background_color='white',
        colormap='RdYlGn' if sentiment == 'positive' else 'RdYlGn_r',
        max_words=120, stopwords=stopwords_set
    ).generate(text)
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(f'{sentiment.capitalize()} Reviews', fontsize=13)

plt.tight_layout()
plt.show()

## 4. NLP Preprocessing

### Pipeline Steps (in order):
1. Lowercase → standardise
2. Expand contractions → 'don't' → 'do not'
3. Remove HTML tags → clean IMDB scraping artifacts
4. Remove URLs → irrelevant
5. Remove emojis → classical ML can't handle Unicode
6. Remove special characters → simplify
7. Remove numbers → low semantic value
8. Remove punctuation → noise
9. Normalise whitespace → clean
10. Tokenise → split into words
11. Remove stopwords → reduce vocabulary
12. Negation handling → preserve 'not good'
13. Lemmatise → 'running' → 'run'

In [ ]:
from src.preprocessing import (
    download_nltk_resources, preprocess_text, preprocess_batch,
    expand_contractions, remove_html_tags, to_lowercase
)

download_nltk_resources()

# Show step-by-step transformation
sample = 'This movie wasn\'t <b>great</b>! I don\'t recommend it. Very, very boring... <br/>'

print('Original:  ', sample)
print('Lowercase: ', to_lowercase(sample))
print('Expand:    ', expand_contractions(to_lowercase(sample)))
print('No HTML:   ', remove_html_tags(expand_contractions(to_lowercase(sample))))
print()
print('Final clean:', preprocess_text(sample))

In [ ]:
# Preprocess all reviews
print('Preprocessing all reviews...')
df['clean_review'] = preprocess_batch(df['review'].tolist())

# Quality check
empty = (df['clean_review'].str.strip() == '').sum()
print(f'Empty after preprocessing: {empty}')
df = df[df['clean_review'].str.strip() != ''].reset_index(drop=True)

# Before vs after
print('\nBefore:', df['review'].iloc[0][:200])
print('\nAfter: ', df['clean_review'].iloc[0][:200])

## 5. Feature Engineering

### TF-IDF Vectorisation
Converts text into numerical vectors ML models can process.

- **TF** (Term Frequency): How often does word appear in this review?
- **IDF** (Inverse Doc Frequency): How rare is this word across all reviews?
- **TF-IDF** = TF × IDF → Common words get penalised, distinctive words are amplified

In [ ]:
from src.feature_engineering import (
    build_tfidf_vectorizer, fit_vectorizer, transform_texts,
    get_top_features
)

texts  = df['clean_review'].tolist()
labels = df['sent_label'].tolist()

vectorizer = build_tfidf_vectorizer(max_features=20000, ngram_range=(1, 2))
vectorizer = fit_vectorizer(vectorizer, texts)

X = transform_texts(vectorizer, texts)
y = df['sent_label'].values

print(f'Feature matrix shape: {X.shape}')
print(f'Sparsity: {1 - X.nnz/(X.shape[0]*X.shape[1]):.4%} of values are zero')
print(f'Vocabulary size: {len(vectorizer.vocabulary_):,}')

In [ ]:
# Top TF-IDF features
features = get_top_features(vectorizer, n=20)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

high_terms, high_scores = zip(*features['high_idf'])
axes[0].barh(list(reversed(high_terms)), list(reversed(high_scores)),
             color='#3498db', alpha=0.8)
axes[0].set_title('Top 20 Features (High IDF = Distinctive)')
axes[0].set_xlabel('IDF Score')

low_terms, low_scores = zip(*features['low_idf'])
axes[1].barh(low_terms, low_scores, color='#e67e22', alpha=0.8)
axes[1].set_title('Top 20 Generic Features (Low IDF = Common)')
axes[1].set_xlabel('IDF Score')

plt.suptitle('TF-IDF Feature Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Model Training

In [ ]:
from src.train import split_data, train_all_models, get_model_configs

X_train, X_test, y_train, y_test = split_data(X, y, test_size=0.20)

print(f'Training set: {X_train.shape[0]:,} reviews')
print(f'Test set    : {X_test.shape[0]:,} reviews')
print(f'Class balance in train: positive={y_train.sum():,}, negative={(y_train==0).sum():,}')

In [ ]:
import time

trained_models = {}
training_times = {}

for name, model in get_model_configs().items():
    t0 = time.perf_counter()
    model.fit(X_train, y_train)
    elapsed = time.perf_counter() - t0
    trained_models[name] = model
    training_times[name] = elapsed
    print(f'✅ {name:<30} trained in {elapsed:.3f}s')

## 7. Model Evaluation

> **Why not just accuracy?**  
> If 95% of reviews are positive, a model that always predicts 'positive' gets 95% accuracy — but it's useless.
> We need Precision, Recall, F1, and ROC-AUC for a complete picture.

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, classification_report
)

all_metrics = {}
all_probs = {}

for name, model in trained_models.items():
    y_pred = model.predict(X_test)
    y_prob = None
    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(X_test)[:, 1]
    elif hasattr(model, 'decision_function'):
        y_prob = model.decision_function(X_test)
    
    metrics = {
        'Accuracy':  accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall':    recall_score(y_test, y_pred, zero_division=0),
        'F1 Score':  f1_score(y_test, y_pred, zero_division=0),
    }
    if y_prob is not None:
        metrics['ROC-AUC'] = roc_auc_score(y_test, y_prob)
        all_probs[name] = (y_test, y_prob)
    all_metrics[name] = metrics

metrics_df = pd.DataFrame(all_metrics).T
print(metrics_df.round(4).to_string())

In [ ]:
# Visual model comparison
fig, ax = plt.subplots(figsize=(13, 5))
x = np.arange(len(metrics_df.columns))
width = 0.25
colors = ['#3498db', '#e74c3c', '#2ecc71']

for i, (model_name, color) in enumerate(zip(metrics_df.index, colors)):
    scores = metrics_df.loc[model_name].values
    offset = (i - len(metrics_df) / 2) * width + width / 2
    bars = ax.bar(x + offset, scores, width,
                  label=model_name.replace('_', ' ').title(),
                  color=color, alpha=0.85, edgecolor='white')
    for bar, score in zip(bars, scores):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{score:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(metrics_df.columns)
ax.set_ylim(0, 1.12)
ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrix for best model (Logistic Regression)
import seaborn as sns

best_model = trained_models['logistic_regression']
y_pred = best_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Confusion Matrix — Logistic Regression', fontsize=14, fontweight='bold')

for ax, data, title in zip(axes, [cm, cm.astype(float)/cm.sum(axis=1, keepdims=True)],
                            ['Raw Counts', 'Normalised']):
    fmt = 'd' if title == 'Raw Counts' else '.2%'
    sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues', ax=ax,
                xticklabels=['Negative','Positive'],
                yticklabels=['Negative','Positive'],
                linewidths=0.5, linecolor='white')
    ax.set_title(title)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.show()

In [ ]:
# ROC Curves
from sklearn.metrics import roc_curve

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#3498db', '#e74c3c', '#2ecc71']

for (name, (y_t, y_p)), color in zip(all_probs.items(), colors):
    fpr, tpr, _ = roc_curve(y_t, y_p)
    auc = roc_auc_score(y_t, y_p)
    ax.plot(fpr, tpr, color=color, linewidth=2.5,
            label=f"{name.replace('_',' ').title()} (AUC={auc:.3f})")

ax.plot([0,1],[0,1],'k--', linewidth=1, label='Random (AUC=0.500)')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate (Recall)')
ax.set_title('ROC Curves — All Models', fontsize=13, fontweight='bold')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Interpretability

What words drive each sentiment prediction? 
Logistic Regression coefficients are directly interpretable — a positive coefficient means the word pushes toward 'positive' sentiment.

In [ ]:
model = trained_models['logistic_regression']
coef  = model.coef_.ravel()
vocab = vectorizer.get_feature_names_out()

top_n = 20
top_pos = np.argsort(coef)[-top_n:][::-1]
top_neg = np.argsort(coef)[:top_n]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Feature Coefficients — Logistic Regression Interpretability',
             fontsize=14, fontweight='bold')

pos_terms = [vocab[i] for i in top_pos]
pos_vals  = [coef[i]  for i in top_pos]
axes[0].barh(pos_terms[::-1], pos_vals[::-1], color='#2ecc71', alpha=0.8)
axes[0].set_title('🟢 Top 20 Positive Sentiment Words')
axes[0].set_xlabel('Coefficient')

neg_terms = [vocab[i] for i in top_neg]
neg_vals  = [coef[i]  for i in top_neg]
axes[1].barh(neg_terms, neg_vals, color='#e74c3c', alpha=0.8)
axes[1].set_title('🔴 Top 20 Negative Sentiment Words')
axes[1].set_xlabel('Coefficient')

plt.tight_layout()
plt.show()

## 9. Inference Demo

The inference pipeline handles: raw text → preprocess → vectorize → predict

In [ ]:
from src.predict import SentimentPredictor, format_prediction, run_demo_predictions

# Note: This requires saved models. Run main.py first, or save models here:
import joblib, os
os.makedirs('../models', exist_ok=True)
for name, model in trained_models.items():
    joblib.dump(model, f'../models/{name}.pkl')
joblib.dump(vectorizer, '../models/tfidf_vectorizer.pkl')
print('Models saved.')

predictor = SentimentPredictor()
run_demo_predictions(predictor)

In [ ]:
# Custom prediction
your_review = "I absolutely loved this film. The acting was phenomenal and the plot kept me hooked throughout."
result = predictor.predict(your_review)
print(format_prediction(result))

---
## 🚀 What's Next?

| Enhancement | Description |
|---|---|
| **BERT / DistilBERT** | State-of-the-art transformer models |
| **Hyperparameter Tuning** | Optuna or GridSearchCV for better accuracy |
| **Ensemble Methods** | Combine multiple models for robustness |
| **Aspect-Based Sentiment** | Analyse sentiment for specific aspects (acting, plot, etc.) |
| **Sarcasm Detection** | Handle ironic/sarcastic reviews |
| **Multilingual** | Support non-English reviews |
| **Real-time API** | Deploy with FastAPI on cloud platforms |